# 01 — Data Cleaning & Harmonization
**HHE Lab Sardinia · Marine Litter Hazard Assessment**

Inputs → `Labenv/Modulo_4_*` (beach) + `Labenv/Modulo_2bis_*` (floating)  
Outputs → `data/processed/beach_litter.csv` · `floating_litter.csv` · `floating_litter_efforts.csv`

**Known schema differences between 2020 and 2023 format:**

| Sheet | 2020 format | 2023 format |
|---|---|---|
| SpiaggiaCamp | `SampleID`, `Lunghezza` | `CodiceCampionamento`, `LunghezzaTransetto` |
| RifiutiCamp | `SampleID`, `IDCategoriaRifiuto`, `NumeroItems` | `CodiceCampionamento`, `JointListCategory`, `NumeroOggetti` |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT   = Path("..").resolve()
LABENV = ROOT / "Labenv"
OUT    = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

---
## 1. Beach Litter — Modulo_4

In [ ]:
MOD4_DIR = LABENV / "Modulo_4_2020_Med_Occ_2018-2023"

SPIAGGIA_RENAME = {
    "Latitudine": "lat", "Longitudine": "lon",
    "NomeSpiaggia": "beach_name", "CodiceSpiaggia": "beach_id",
    "Regione": "region", "Comune": "municipality"
}
CAMP_RENAME = {
    "SampleID": "survey_id", "CodiceCampionamento": "survey_id",
    "Year": "year", "Anno": "year",
    "Month": "month", "Mese": "month",
    "Day": "day", "Giorno": "day",
    "LunghezzaTransetto": "transect_m", "Lunghezza": "transect_m",
    "CodiceSpiaggia": "beach_id", "CodiceStagione": "season"
}
RIFIUTI_RENAME = {
    "SampleID": "survey_id", "CodiceCampionamento": "survey_id",
    "JointListCategory": "category", "IDCategoriaRifiuto": "category",
    "NumeroOggetti": "n_items", "NumeroItems": "n_items",
    "Sorgente": "source"
}

def norm(df, rename, keep):
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    return df[[c for c in keep if c in df.columns]]

beach_frames = []
for f in sorted(MOD4_DIR.glob("*.xl*")):
    engine = "xlrd" if f.suffix == ".xls" else "openpyxl"
    try:
        xl       = pd.ExcelFile(f, engine=engine)
        spiaggia = norm(xl.parse("Spiaggia"), SPIAGGIA_RENAME,
                        ["beach_id","beach_name","lat","lon","region","municipality"])
        camp     = norm(xl.parse("SpiaggiaCamp"), CAMP_RENAME,
                        ["survey_id","beach_id","year","month","day","season","transect_m"])
        rifiuti  = norm(xl.parse("RifiutiCamp"), RIFIUTI_RENAME,
                        ["survey_id","category","n_items","source"])
    except Exception as e:
        print(f"ERROR {f.name}: {e}"); continue

    items = (rifiuti.groupby("survey_id")["n_items"].sum()
             .reset_index().rename(columns={"n_items": "total_items"}))
    df = camp.merge(items, on="survey_id", how="left")
    df = df.merge(spiaggia, on="beach_id", how="left")
    df["transect_m"] = pd.to_numeric(df["transect_m"], errors="coerce").fillna(100)
    df["items_per_100m"] = df["total_items"] / df["transect_m"] * 100
    beach_frames.append(df)

beach_df = pd.concat(beach_frames, ignore_index=True)
beach_df = beach_df.dropna(subset=["year", "total_items"])
beach_df["year"] = beach_df["year"].astype(int)

# Fill missing lat/lon from known coords per beach_id
coords = beach_df.dropna(subset=["lat","lon"]).groupby("beach_id")[["lat","lon"]].first()
for idx, row in beach_df[beach_df["lat"].isna()].iterrows():
    if row["beach_id"] in coords.index:
        beach_df.at[idx, "lat"] = coords.loc[row["beach_id"], "lat"]
        beach_df.at[idx, "lon"] = coords.loc[row["beach_id"], "lon"]

beach_df.to_csv(OUT / "beach_litter.csv", index=False)
print(f"Beach: {len(beach_df)} surveys · {beach_df['beach_id'].nunique()} beaches · years {sorted(beach_df['year'].unique())}")
print(f"items/100m: median={beach_df['items_per_100m'].median():.1f} · max={beach_df['items_per_100m'].max():.1f}")
beach_df.head()

---
## 2. Floating Litter — Modulo_2bis

In [ ]:
MOD2BIS_DIR = LABENV / "Modulo_2bis_2020_Med_Occ_2018-2023"

STAZ_RENAME = {
    "NationalStationID": "station_id", "NationalStationName": "station_name",
    "Latitude": "lat_station", "Longitude": "lon_station",
    "Region": "region", "SeaDepth": "depth_m"
}
MACROFLOT_RENAME = {
    "NationalStationID": "station_id", "COD_Effort": "effort_id",
    "ID_Transect": "transect_id", "Survey_N": "survey_n",
    "Latitude": "lat", "Longitude": "lon",
    "Day": "day", "Month": "month", "Year": "year",
    "Material": "material", "IDCategoriaRifiuto": "category",
    "Size": "size", "Buoyancy": "buoyancy", "Source_lit": "source",
    "Strip": "strip_width_m", "Mean_speed": "speed_knots",
    "Time_effective": "time_obs"  # clock time of sighting, not duration
}
MACROFLOT_KEEP = [
    "station_id","effort_id","transect_id","survey_n",
    "lat","lon","day","month","year",
    "material","category","size","buoyancy","source",
    "strip_width_m","speed_knots","time_obs"
]

# Normalize material names IT→EN
MAT_MAP = {
    "Polimeri artificiali": "Artificial polymer",
    "Materiale naturale":   "Natural matter",
    "Alghe/Fanerogame":     "Natural matter",
    "Legno lavorato":       "Processed wood",
    "Rifiuto alimentare":   "Food waste",
    "Gomma":                "Rubber",
    "Metallo":              "Metal",
    "Vetro/ceramica":       "Glass/ceramics",
    "Tessile":              "Textiles",
    "Carta/cartone":        "Paper/cardboard",
}

float_frames = []
for f in sorted(MOD2BIS_DIR.glob("*.xl*")):
    engine = "xlrd" if f.suffix == ".xls" else "openpyxl"
    try:
        xl       = pd.ExcelFile(f, engine=engine)
        stazioni = norm(xl.parse("Stazioni"), STAZ_RENAME,
                        ["station_id","station_name","lat_station","lon_station","region","depth_m"])
        mf       = norm(xl.parse("MacroFlotCamp"), MACROFLOT_RENAME, MACROFLOT_KEEP)
    except Exception as e:
        print(f"ERROR {f.name}: {e}"); continue
    df = mf.merge(stazioni, on="station_id", how="left")
    float_frames.append(df)

float_df = pd.concat(float_frames, ignore_index=True)
float_df["material"] = float_df["material"].replace(MAT_MAP)
float_df["year"]  = pd.to_numeric(float_df["year"],  errors="coerce").astype("Int64")
float_df["month"] = pd.to_numeric(float_df["month"], errors="coerce").astype("Int64")
float_df.to_csv(OUT / "floating_litter.csv", index=False)

print(f"Floating: {len(float_df)} obs · {float_df['station_id'].nunique()} stations · years {sorted(float_df['year'].dropna().unique())}")
float_df.head()

In [ ]:
# Effort-level aggregation: n_items + area per transect effort
# time_obs is sighting clock time → effort duration = max-min per effort_id
float_df["time_dt"] = pd.to_datetime(float_df["time_obs"], format="%H:%M:%S", errors="coerce")
effort_dur = (
    float_df.dropna(subset=["time_dt"])
    .groupby("effort_id")["time_dt"]
    .agg(lambda x: (x.max() - x.min()).total_seconds() / 3600)
    .reset_index().rename(columns={"time_dt": "duration_h"})
)
effort_meta = (
    float_df.groupby("effort_id").agg(
        n_items=("effort_id","count"),
        strip_width_m=("strip_width_m","mean"),
        speed_knots=("speed_knots","mean"),
        year=("year","first"), month=("month","first"),
        station_id=("station_id","first"),
        lat_station=("lat_station","first"),
        lon_station=("lon_station","first"),
        region=("region","first")
    ).reset_index()
)
effort_meta = effort_meta.merge(effort_dur, on="effort_id", how="left")
effort_meta["area_km2"] = (
    effort_meta["strip_width_m"] *
    effort_meta["speed_knots"] * 1852 *
    effort_meta["duration_h"]
) / 1e6
effort_meta["density_km2"] = effort_meta["n_items"] / effort_meta["area_km2"].replace(0, np.nan)
effort_meta.to_csv(OUT / "floating_litter_efforts.csv", index=False)
print(f"Efforts: {len(effort_meta)}")
effort_meta[["n_items","area_km2","density_km2"]].describe().round(3)

---
## 3. Coverage Summary

In [ ]:
print("=== BEACH LITTER ===")
print(f"  Surveys : {len(beach_df)}")
print(f"  Beaches : {beach_df['beach_id'].nunique()}")
print(f"  Years   : {sorted(beach_df['year'].unique())}")
print(f"  items/100m: {beach_df['items_per_100m'].describe().round(1).to_dict()}")

print("\n=== FLOATING LITTER ===")
print(f"  Observations: {len(float_df)}")
print(f"  Efforts     : {len(effort_meta)}")
print(f"  Stations    : {float_df['station_id'].nunique()}")
print(f"  Years       : {sorted(float_df['year'].dropna().unique())}")
print(f"  % Plastic   : {(float_df['material']=='Artificial polymer').mean()*100:.1f}%")

# Survey heatmap: beach × year
pivot = beach_df.pivot_table(index="beach_id", columns="year",
                              values="items_per_100m", aggfunc="mean").round(1)
pivot